In [1]:
import re
import pandas as pd

# 输入文件名，请根据你的实际文件名修改
input_file = "/share/home/zhoujb/project/riceDataComb/processData/dataFromCowpea/cowpea_signals_raw.xlsx"
output_file = "/share/home/zhoujb/project/riceDataComb/processData/dataFromCowpea/gene_list.txt"

In [2]:
# 读取文件
if input_file.endswith((".xlsx", ".xls")):
    df = pd.read_excel(input_file)
elif input_file.endswith(".csv"):
    df = pd.read_csv(input_file)
else:
    df = pd.read_csv(input_file, sep="\t")

# 确保 start 和 end 是数值型
df["start"] = pd.to_numeric(df["start"], errors="coerce")
df["end"] = pd.to_numeric(df["end"], errors="coerce")

# 删除 Chr、start、end 任意为空的行
df = df.dropna(subset=["Chr", "start", "end"]).copy()

# 根据 Chr、start、end 去重
df = df.drop_duplicates(subset=["Chr", "start", "end"], keep="first").copy()

# 提取 Chr 中的数字，用于自然排序，例如 Chr1, Chr2, Chr10
def chr_sort_key(x):
    x = str(x)
    m = re.search(r"\d+", x)
    return int(m.group()) if m else 999999

df["_chr_order"] = df["Chr"].apply(chr_sort_key)

# 排序：先按 Chr，再按 start
df = df.sort_values(
    by=["_chr_order", "start", "end"],
    ascending=[True, True, True]
).drop(columns=["_chr_order"])

# start/end 如果本来是整数，保存前转回整数
df["start"] = df["start"].astype(int)
df["end"] = df["end"].astype(int)

# 保存为 txt，制表符分隔
df.to_csv(output_file, sep="\t", index=False)

print(f"完成！去重排序后的文件已保存为：{output_file}")
print(f"最终保留区间数：{df.shape[0]}")

完成！去重排序后的文件已保存为：/share/home/zhoujb/project/riceDataComb/processData/dataFromCowpea/gene_list.txt
最终保留区间数：91
